We want the KODIS dataset to be converted to this a csv to this format:

In [9]:
import pandas as pd

df = pd.DataFrame([{"kodis-id": 123, "buyer or seller": "buyer", "concatenated utterances": "Hello my name is Kaleen. I want to make an offer. What do you mean by that?"}, {"kodis-id": 123, "buyer or seller": "seller", "concatenated utterances": "Hi my name is Sarah. I don't want to give you a refund. I mean no refund"}, ])
df.to_csv("example.csv")
df

,kodis-id,buyer or seller,concatenated utterances
0,123,buyer,Hello my name is Kaleen. I want to make an off...
1,123,seller,Hi my name is Sarah. I don't want to give you ...


# Testing helper functions

In [31]:
import pandas as pd
kodis_df = pd.read_csv("DO-NOT-DISTRIBUTE-KODIS-human-human-subset.csv")
kodis_df.columns

Index(['Unnamed: 0', 'b_StartDate', 'b_EndDate', 'b_Status', 'b_Progress',
       'b_Duration (in seconds)', 'b_Finished', 'b_RecordedDate',
       'b_ResponseId', 'b_RecipientLastName',
       ...
       's_points_binary_apol', 'b_points_binary_apol', 's_points_4level_apol',
       'b_points_4level_apol', 's_points_3level_apol', 'b_points_3level_apol',
       'joint_points_binary_apol', 'joint_points_3level_apol',
       'joint_points_4level_apol', 'Integrative_Potential_COSINE'],
      dtype='object', length=419)

Remove the following utterances:
- "I Walk Away"
- "Submitted agreement: ..."
- "Accept Deal"
- "Reject Deal"

These are preset values that the participants can click, and not utterances they type in, so we want to remove them from the LIWC analysis.

In [32]:
IGNORE_PHRASES = {"I Walk Away", "Submitted agreement:", "Accept Deal", "Reject Deal"}
BUYER = "BUYER"
SELLER = "SELLER"

In [33]:
data = []

for index, row in kodis_df.iterrows():
    row_number = index  # Row number (index)
    kodis_id = row['Unnamed: 0']  # The column "Unnamed: 0" is the kodis-id
    conv = kodis_df['s_fullChat'][row_number].replace("<b>","").replace("</b>","").replace("<br>","\n").replace("[Other]","BUYER").replace("[You]","SELLER")

    buyer_turns, seller_turns = [],[]

    for turn in conv.split("\n"):
        content = turn[8:] if turn.startswith(SELLER) else turn[7:]
        if any(content.startswith(phrase) for phrase in IGNORE_PHRASES):
            continue

        if turn.startswith(BUYER):
            buyer_turns.append(content)
        else:
            seller_turns.append(content)

    data.append({"kodis-id": kodis_id, "buyer or seller": "buyer", "concatenated utterances": " ".join(buyer_turns)})
    data.append({"kodis-id": kodis_id, "buyer or seller": "seller", "concatenated utterances": " ".join(seller_turns)})    

df = pd.DataFrame(data)
print(df)

# Save to CSV
df.to_csv("processed_conversations.csv", index=False)

     kodis-id buyer or seller  \
0        2523           buyer   
1        2523          seller   
2        2525           buyer   
3        2525          seller   
4        2526           buyer   
..        ...             ...   
435      2879          seller   
436      2881           buyer   
437      2881          seller   
438      2889           buyer   
439      2889          seller   

                               concatenated utterances  
0    I would like a refund on the kobe bryant jerse...  
1    Unfortunately that is not true. All our mercha...  
2    Hello. I would like to reach an agreement here...  
3    Hello, can you tell me more about the issue th...  
4    I would like a refund In that case, I would li...  
..                                                 ...  
435  it was what you ordered for i got delivered to...  
436  Hey, I received my order and I'm not happy at ...  
437  Can you explain the issue please? ok, let me h...  
438  Hi there, thank you for send